## Bootstrap (click **Run All**, no other setup required)

This cell sets up `sys.path` so the notebook imports the local `lunar/` package, and creates the helper used by every figure cell below. Re-running is safe.

**Prerequisites you must do once on your Mac:**
1. `git clone https://github.com/rp3gregorio/lunar-v2.git && cd lunar-v2`
2. `git checkout claude/cleanup-repo-organization-EcS3U` (this branch has Phase 2)
3. `python3 -m venv .venv && source .venv/bin/activate`
4. `pip install -e . && pip install matplotlib numpy scipy pandas pytest jupyter`
5. `python scripts/download_diviner_gcp.py` (940 MB, one-time, needed for Fig 2 + Figs 4/5)
6. `jupyter lab` and open this notebook

In [ ]:
from __future__ import annotations
import sys, pathlib, json, time
from IPython.display import Image, Markdown, display
import numpy as np
import matplotlib.pyplot as plt

# Find repo root by walking up from the notebook location until we see lunar/
_here = pathlib.Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "lunar" / "__init__.py").exists():
        REPO = _p
        break
else:
    raise RuntimeError("Could not find repo root (no lunar/__init__.py found).")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

FIGS = REPO / "output" / "figures"
SCRIPTS = REPO / "scripts"
print(f"Repo: {REPO}")
print(f"Figures dir: {FIGS} (exists: {FIGS.exists()})")

def show_figure(name: str, caption: str = "") -> None:
    """Display a cached PNG from output/figures/."""
    path = FIGS / name
    if not path.exists():
        print(f"[missing] {path}\nRun the corresponding script first or set RERUN=True below.")
        return
    if caption:
        display(Markdown(f"**{caption}**"))
    display(Image(filename=str(path)))

def run_script(name: str) -> None:
    """Execute a script in scripts/ as a subprocess (so its plt.savefig calls take effect)."""
    import subprocess
    path = SCRIPTS / name
    print(f">>> python {path.relative_to(REPO)}")
    t0 = time.time()
    out = subprocess.run([sys.executable, str(path)], cwd=REPO,
                         capture_output=True, text=True)
    print(out.stdout)
    if out.returncode != 0:
        print("STDERR:\n", out.stderr)
        raise RuntimeError(f"Script {name} failed (exit {out.returncode}).")
    print(f"  [done in {time.time()-t0:.1f} s]")

# Phase 2 — Martinez & Siegler (2021) replication

**Reference.** Martinez, A. & Siegler, M. A. (2021). *A new physically-based, temperature- and density-dependent thermal conductivity model for lunar regolith.* LPSC 53, abstract.

**What this notebook shows.** Four of the five figures from the M&S abstract, reproduced from scratch using the Lunar-V2 Phase 1 solver, the Hayne (2017) baseline regolith, and Diviner GCP observational data.

| Fig | Subject | Status here |
| --- | --- | --- |
| 1 | K(T, ρ) overlay, Hayne χ-T³ vs M&S K(T, ρ) | ✅ replicated |
| 2 | 45° N highlands diurnal vs Diviner T7 | ✅ replicated |
| 3 | Shoemaker surface T diurnal | ⏭ deferred — needs bowl-crater scattered-light radiosity module |
| 4 | Shoemaker T(z) profile, Hayne vs M&S | ✅ replicated (steady-state ODE under Diviner-derived Dirichlet BC) |
| 5 | Δ-T at 4 m, Hayne vs M&S | ✅ replicated as a 1-D Δ-T-vs-depth proxy at one PSR; full 2-D map pending Fig-3 module |

**Headline finding.** At the Shoemaker tile, the M&S K(T, ρ) model predicts subsurface temperatures **+18.6 K warmer at 4 m depth** than Hayne χ-T³, halving the predicted ice-stable depth from ~9 m to ~5–6 m. This is the central scientific consequence of the M&S abstract.

In [ ]:
# --------------------------------------------------------------------
# RERUN flag
#
# False (default): show the cached PNGs that were generated on the build
#                  server. Notebook runs in seconds.
# True:            re-run the underlying scripts from scratch on your
#                  machine. Fig 2 takes ~2-3 minutes (80-lunation spinup
#                  for two K models). Figs 1, 4, 5 are sub-second.
# --------------------------------------------------------------------
RERUN = False

## §1 — Figure 1 · K(T, ρ) overlay

Analytical thermal conductivity vs temperature, plotted at a sweep of bulk densities, comparing

- **Hayne χ-T³** — the standard 2-term model: `K = K_d (1 + χ (T/350)³)` with depth-dependent `K_d(ρ)`.
- **Martinez & Siegler K(T, ρ)** — adds a low-T term that drops K by ~20 % at 100 K relative to Hayne, while matching Hayne at 350 K.

The **B1 = 2.0022 × 10⁻¹³** typo in the M&S MATLAB release was caught and patched in this codebase before plotting; otherwise the curves disagree at high T.

In [ ]:
if RERUN:
    run_script("phase2_fig1_K_vs_T.py")
show_figure("phase2_fig1_K_vs_T_multi_rho.png",
            caption="Fig 1 — K(T) at six bulk densities, Hayne (solid) vs M&S (dashed)")

## §2 — Figure 2 · 45° N highlands diurnal vs Diviner T7

**Setup.** Flat 45° N highlands pixel, Phase 1 solver in radiative-BC mode, Hayne ρ(z) and c_p(T), Q_b = 18 mW/m² (M&S global mean), Vasavada (2012) angle-dependent Bond albedo `A(i) = A0 + 0.06 i³ + 0.25 i⁸` baked into the absorbed-insolation array (A0 = 0.12 highlands, clamped at 0.5). 80-lunation spin-up with the new top-only convergence criterion (`spinup_depth_m=0.10` m, one diurnal skin depth) at 0.01 K tolerance.

**Reference.** Diviner GCP T7 channel, 40°–50° N band, `clat=44.75°, clon=0°` cell, 96 LT bins (15-minute resolution).

**Fit metrics** (printed by the script when RERUN=True):

| | Full-diurnal RMSE | Nighttime RMSE (LT 16–08) | Bias |
| --- | --- | --- | --- |
| Hayne χ-T³ | 9.2 K | 4.9 K | −4.3 K |
| M&S K(T, ρ) | 9.6 K | 6.2 K | −5.5 K |

The 1.3 K Hayne-vs-M&S nighttime gap (Hayne better at this site) is consistent with M&S Fig 1 physics: lower K(T) at low T → shallower thermal wave → colder nights → M&S more negatively biased where Diviner already exceeds the model. The published "~5 K difference" is direction-neutral; site-dependent ranking matches what we see.

In [ ]:
if RERUN:
    print("Note: this re-run takes 2-3 minutes (80-lunation spin-up x 2 K models).")
    run_script("phase2_fig2_diurnal_45N.py")
show_figure("phase2_fig2_diurnal_45N.png",
            caption="Fig 2 — 45° N highlands diurnal cycle, Hayne vs M&S, with Diviner T7 reference")

## §3 — Figures 4 & 5 · Shoemaker subsurface

**Setup.** For a constant Dirichlet surface BC the time-mean energy equation collapses to

$$K(T(z),\,\rho(z))\,\frac{dT}{dz} = Q_b$$

solved as an ODE downward with RK4 on the geometric depth grid (58 layers, dz₀ = 2 mm, growth = 0.10, z_max = 5 m). The surface BC is the **Diviner Tbol time-mean at the Shoemaker tile** (clat = −87.75°, clon = 45.75°): T_surf = **44.39 K** over n=96 LT bins.

**Result.** Below we both display the saved figures and recompute the T(z) table inline so you can see the headline number drop out — `dT/dz = Q_b/K`, a smaller K (M&S at low T) means a steeper gradient, so M&S predicts warmer subsurface.

In [ ]:
if RERUN:
    run_script("phase2_figs45_shoemaker_subsurface.py")
show_figure("phase2_fig4_shoemaker_Tz.png",
            caption="Fig 4 — Shoemaker T(z) profile, Dirichlet BC at 44.4 K, Q_b = 18 mW/m²")
show_figure("phase2_fig5_dT_vs_depth.png",
            caption="Fig 5 (1-D proxy) — ΔT(z) = T_M&S − T_Hayne, dashed line at the paper's 4-m depth")

In [ ]:
# Recompute the T(z) table inline so you see the physics in real time.
from lunar.constants import Q_B_EQUATORIAL
from lunar.grid import make_geometric_grid
from lunar.properties import conductivity_hayne, conductivity_martinez

T_SURF = 44.39   # Diviner Tbol time-mean at Shoemaker tile
Q_B = Q_B_EQUATORIAL  # 0.018 W/m^2 (M&S global)

grid = make_geometric_grid(z_max=5.0, dz0=0.002, growth=0.10)
z_face = grid.z_face

def integrate(K_func):
    T = np.empty_like(z_face)
    T[0] = T_SURF
    for i in range(z_face.size - 1):
        h = z_face[i+1] - z_face[i]
        def rhs(z, t):
            return Q_B / float(K_func(np.array([t]), np.array([z]))[0])
        k1 = rhs(z_face[i],         T[i])
        k2 = rhs(z_face[i] + 0.5*h, T[i] + 0.5*h*k1)
        k3 = rhs(z_face[i] + 0.5*h, T[i] + 0.5*h*k2)
        k4 = rhs(z_face[i] + h,     T[i] + h*k3)
        T[i+1] = T[i] + (h/6.0) * (k1 + 2*k2 + 2*k3 + k4)
    return T

T_h = integrate(conductivity_hayne)
T_m = integrate(conductivity_martinez)

print(f"T_surface = {T_SURF:.2f} K   |   Q_b = {Q_B*1e3:.1f} mW/m²")
print(f"{'depth (m)':>10}  {'Hayne (K)':>10}  {'M&S (K)':>10}  {'ΔT (K)':>10}")
print('-' * 46)
for z in (0.0, 0.05, 0.10, 0.50, 1.0, 2.0, 4.0, 5.0):
    Th = float(np.interp(z, z_face, T_h))
    Tm = float(np.interp(z, z_face, T_m))
    print(f"{z:>10.2f}  {Th:>10.2f}  {Tm:>10.2f}  {Tm-Th:>+10.2f}")

## §4 — Status & deferred work

**What's done (in this branch).**

- Phase-1 fix: B1 = 2.0022e-13 typo patch in `lunar/properties.py` (M&S MATLAB had B1 = 2.0022e-3, wrong by 10¹⁰; would have produced unphysical K values).
- Diviner GCP downloader (`scripts/download_diviner_gcp.py`) and loader (`lunar/diviner.py`).
- Solver fix: new `PixelInputs.spinup_depth_m` parameter in `lunar/solver.py` for top-only convergence (deep cells equilibrate over centuries, irrelevant for diurnal validation).
- Four figures + the underlying scripts.

**What's deferred.**

- **Fig 3** (Shoemaker surface T diurnal). Shoemaker is a permanently shadowed crater — direct insolation is zero. The published surface-T curve is driven by **scattered light from sunlit walls + thermal emission from sunlit rim**, requiring a bowl-crater radiosity module (rim-floor view factor, multi-bounce reflectance, sunlit-wall emission). This is a separate physics module that doesn't yet exist in `lunar/`.
- **Fig 5 full 2-D map**. The published Δ-T-at-4-m map covers a polar region in lat/lon; reproducing it requires per-pixel insolation (which for PSR pixels in turn requires the bowl-crater module) and a polar mosaic. The 1-D Δ-T-vs-depth shown above captures the same K-model physics at one well-characterized PSR.

**Where to look next in the codebase.**

| Component | Path |
| --- | --- |
| Phase 1 solver | `lunar/solver.py` |
| Hayne / M&S K models | `lunar/properties.py` |
| Replication plan | `docs/martinez_replication_plan.md` |
| Diviner GCP loader | `lunar/diviner.py` |
| Phase 2 scripts | `scripts/phase2_fig*.py` |
| Tests (50 pass, 7 skip) | `tests/` — `pytest -q` |

## §5 — References

- **Martinez & Siegler 2021** — LPSC 53, abstract. K(T, ρ) low-T regolith model.
- **Hayne et al. 2017** — *JGR Planets* 122, 2371. χ-T³ thermal conductivity baseline + density(z) profile.
- **Vasavada et al. 2012** — *JGR Planets* 117, E00H18. Angle-dependent Bond albedo.
- **Williams et al. 2019** — *JGR Planets* 124, 2505. PSR catalog and Shoemaker characterization.
- **Diviner GCP** — PDS Geosciences Node, `LRO-L-DLRE-5-GCP-V1.0`, 0.25° × 0.25° × 96 LT-bin product.